# FinOps Cloud Cost Forecasting — MLflow Experiment Tracking

This notebook rebuilds the MLflow experiment using only the genuine FinOps
forecasting dataset and regression models.

Pipeline:

1. Mount Google Drive
2. Locate and validate the processed hourly dataset
3. Reconstruct the forecasting features
4. Create chronological train/validation/test splits
5. Train and evaluate regression models
6. Configure persistent MLflow tracking
7. Log genuine metrics and model artifacts
8. Compare candidates against the naive baseline
9. Apply the model-promotion gate

In [ ]:
# ============================================================
# CELL 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
# ============================================================
# CELL 2 — LOCATE AND VALIDATE THE REAL HOURLY DATASET
# ============================================================

from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/bitbrains_processed"
)

print("Processed directory exists:", PROCESSED_DIR.exists())

# Display available CSV files
csv_files = sorted(PROCESSED_DIR.glob("*.csv"))

print("\nCSV files found:")
for file_path in csv_files:
    print("-", file_path.name)

# Use the cleaned 719-row hourly dataset
FINOPS_DATA_PATH = (
    PROCESSED_DIR
    / "bitbrains_hourly_cost_timeseries_clean.csv"
)

if not FINOPS_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Expected dataset not found:\n{FINOPS_DATA_PATH}"
    )

# Load genuine FinOps hourly data
hourly_finops_raw = pd.read_csv(
    FINOPS_DATA_PATH,
    parse_dates=["time_bucket"]
)

print("\nSelected dataset:")
print(FINOPS_DATA_PATH)

print("\nDataset shape:")
print(hourly_finops_raw.shape)

print("\nColumns:")
print(hourly_finops_raw.columns.tolist())

print("\nTime range:")
print("Start:", hourly_finops_raw["time_bucket"].min())
print("End  :", hourly_finops_raw["time_bucket"].max())

print("\nFirst 3 rows:")
display(hourly_finops_raw.head(3))

print("\nMissing values:")
display(hourly_finops_raw.isna().sum().to_frame("missing_count"))

Processed directory exists: True

CSV files found:
- bitbrains_5min_resource_timeseries.csv
- bitbrains_hourly_cost_timeseries.csv
- bitbrains_hourly_cost_timeseries_clean.csv
- bitbrains_vm_features.csv
- bitbrains_vm_summary.csv

Selected dataset:
/content/drive/MyDrive/bitbrains_processed/bitbrains_hourly_cost_timeseries_clean.csv

Dataset shape:
(719, 8)

Columns:
['time_bucket', 'cpu_mean', 'memory_mean_gb', 'disk_activity_kbps', 'network_activity_kbps', 'active_vms', 'resource_cost_index', 'estimated_cost_index']

Time range:
Start: 2013-08-12 14:00:00+00:00
End  : 2013-09-11 12:00:00+00:00

First 3 rows:


,time_bucket,cpu_mean,memory_mean_gb,disk_activity_kbps,network_activity_kbps,active_vms,resource_cost_index,estimated_cost_index
0,2013-08-12 14:00:00+00:00,5.424531,0.571483,257.485959,89.581368,920.583333,0.160205,24.397998
1,2013-08-12 15:00:00+00:00,8.042373,0.682148,251.989447,100.663045,927.416667,0.248989,38.996738
2,2013-08-12 16:00:00+00:00,11.640584,0.710577,224.932377,55.266386,925.166667,0.349348,55.498785



Missing values:


,missing_count
time_bucket,0
cpu_mean,0
memory_mean_gb,0
disk_activity_kbps,0
network_activity_kbps,0
active_vms,0
resource_cost_index,0
estimated_cost_index,0


In [ ]:
# ============================================================
# CELL 3 — VALIDATE HOURLY DATASET INTEGRITY
# ============================================================

import numpy as np
import pandas as pd

EXPECTED_COLUMNS = [
    "time_bucket",
    "cpu_mean",
    "memory_mean_gb",
    "disk_activity_kbps",
    "network_activity_kbps",
    "active_vms",
    "resource_cost_index",
    "estimated_cost_index"
]

NUMERIC_COLUMNS = EXPECTED_COLUMNS[1:]

# Create the working copy and ensure chronological order
hourly_finops = (
    hourly_finops_raw
    .sort_values("time_bucket")
    .reset_index(drop=True)
    .copy()
)

# Individual validation checks
columns_correct = (
    hourly_finops.columns.tolist() == EXPECTED_COLUMNS
)

row_count_correct = len(hourly_finops) == 719

duplicate_timestamps = int(
    hourly_finops["time_bucket"].duplicated().sum()
)

missing_cells = int(
    hourly_finops.isna().sum().sum()
)

non_numeric_cells = int(
    hourly_finops[NUMERIC_COLUMNS]
    .apply(pd.to_numeric, errors="coerce")
    .isna()
    .sum()
    .sum()
)

non_finite_cells = int(
    (~np.isfinite(hourly_finops[NUMERIC_COLUMNS])).sum().sum()
)

negative_values = int(
    (hourly_finops[NUMERIC_COLUMNS] < 0).sum().sum()
)

time_differences = (
    hourly_finops["time_bucket"].diff().dropna()
)

non_hourly_intervals = int(
    (time_differences != pd.Timedelta(hours=1)).sum()
)

validation_results = pd.DataFrame({
    "Check": [
        "Expected columns",
        "Expected 719 rows",
        "Duplicate timestamps",
        "Missing cells",
        "Non-numeric cells",
        "Infinite values",
        "Negative numeric values",
        "Non-hourly intervals"
    ],
    "Result": [
        "PASS" if columns_correct else "FAIL",
        "PASS" if row_count_correct else "FAIL",
        "PASS" if duplicate_timestamps == 0 else "FAIL",
        "PASS" if missing_cells == 0 else "FAIL",
        "PASS" if non_numeric_cells == 0 else "FAIL",
        "PASS" if non_finite_cells == 0 else "FAIL",
        "PASS" if negative_values == 0 else "FAIL",
        "PASS" if non_hourly_intervals == 0 else "FAIL"
    ],
    "Observed": [
        len(hourly_finops.columns),
        len(hourly_finops),
        duplicate_timestamps,
        missing_cells,
        non_numeric_cells,
        non_finite_cells,
        negative_values,
        non_hourly_intervals
    ]
})

print("FINOPS HOURLY DATASET VALIDATION")
print("=" * 55)

display(validation_results)

print("\nNumeric summary:")
display(
    hourly_finops[NUMERIC_COLUMNS]
    .describe()
    .T
)

all_checks_passed = (
    validation_results["Result"] == "PASS"
).all()

if not all_checks_passed:
    raise ValueError(
        "Dataset validation failed. Do not continue."
    )

print("\nAll validation checks passed.")
print("Dataset is ready for feature engineering.")

FINOPS HOURLY DATASET VALIDATION


,Check,Result,Observed
0,Expected columns,PASS,8
1,Expected 719 rows,PASS,719
2,Duplicate timestamps,PASS,0
3,Missing cells,PASS,0
4,Non-numeric cells,PASS,0
5,Infinite values,PASS,0
6,Negative numeric values,PASS,0
7,Non-hourly intervals,PASS,0



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
cpu_mean,719.0,7.853959,3.204986,2.072840,5.144935,7.254856,11.053050,15.305289
memory_mean_gb,719.0,0.575186,0.238456,0.212679,0.374652,0.547609,0.717053,1.562801
disk_activity_kbps,719.0,405.257640,148.823144,216.130126,317.829409,354.338639,429.438475,983.792217
network_activity_kbps,719.0,118.758381,58.449818,32.609853,75.568872,105.686699,146.764766,503.163005
active_vms,719.0,924.009852,27.216751,593.500000,911.333333,924.166667,943.416667,965.250000
resource_cost_index,719.0,0.244246,0.119979,0.019946,0.144331,0.232434,0.352188,0.536530
estimated_cost_index,719.0,38.216820,19.728088,1.335226,21.787790,36.274521,55.965722,86.277015



All validation checks passed.
Dataset is ready for feature engineering.


In [ ]:
# ============================================================
# CELL 4 — FEATURE ENGINEERING
# ============================================================

import numpy as np

finops_features_df = hourly_finops.copy()

# ------------------------------------------------------------
# Calendar features known at prediction time
# ------------------------------------------------------------

finops_features_df["hour"] = (
    finops_features_df["time_bucket"].dt.hour
)

finops_features_df["day_of_week"] = (
    finops_features_df["time_bucket"].dt.dayofweek
)

finops_features_df["day_of_month"] = (
    finops_features_df["time_bucket"].dt.day
)

finops_features_df["is_weekend"] = (
    finops_features_df["day_of_week"] >= 5
).astype(int)

# Cyclical time encoding
finops_features_df["hour_sin"] = np.sin(
    2 * np.pi * finops_features_df["hour"] / 24
)

finops_features_df["hour_cos"] = np.cos(
    2 * np.pi * finops_features_df["hour"] / 24
)

finops_features_df["dow_sin"] = np.sin(
    2 * np.pi * finops_features_df["day_of_week"] / 7
)

finops_features_df["dow_cos"] = np.cos(
    2 * np.pi * finops_features_df["day_of_week"] / 7
)

# ------------------------------------------------------------
# Hour-to-hour change features
# ------------------------------------------------------------

finops_features_df["cpu_change"] = (
    finops_features_df["cpu_mean"].diff()
)

finops_features_df["memory_change"] = (
    finops_features_df["memory_mean_gb"].diff()
)

finops_features_df["disk_change"] = (
    finops_features_df["disk_activity_kbps"].diff()
)

finops_features_df["network_change"] = (
    finops_features_df["network_activity_kbps"].diff()
)

finops_features_df["active_vms_change"] = (
    finops_features_df["active_vms"].diff()
)

finops_features_df["resource_cost_change"] = (
    finops_features_df["resource_cost_index"].diff()
)

# ------------------------------------------------------------
# Historical cost feature
# ------------------------------------------------------------

finops_features_df["cost_lag_1"] = (
    finops_features_df["estimated_cost_index"].shift(1)
)

# ------------------------------------------------------------
# One-hour-ahead forecasting target
# ------------------------------------------------------------

finops_features_df["target_next_hour_cost"] = (
    finops_features_df["estimated_cost_index"].shift(-1)
)

# Remove first row affected by diff/lag and last row without target
finops_modeling_df = (
    finops_features_df
    .dropna()
    .reset_index(drop=True)
)

print("FEATURE ENGINEERING COMPLETE")
print("=" * 55)

print("Original hourly rows :", len(hourly_finops))
print("Modeling rows        :", len(finops_modeling_df))
print("Total columns        :", finops_modeling_df.shape[1])
print("Missing cells        :", finops_modeling_df.isna().sum().sum())

print("\nModeling time range:")
print("Start:", finops_modeling_df["time_bucket"].min())
print("End  :", finops_modeling_df["time_bucket"].max())

print("\nTarget statistics:")
display(
    finops_modeling_df[
        "target_next_hour_cost"
    ].describe().to_frame("target")
)

print("\nFeature verification:")
display(
    finops_modeling_df[[
        "time_bucket",
        "estimated_cost_index",
        "cost_lag_1",
        "cpu_change",
        "target_next_hour_cost"
    ]].head(5)
)

# Strict checks
assert len(finops_modeling_df) == 717
assert finops_modeling_df.shape[1] == 24
assert finops_modeling_df.isna().sum().sum() == 0

print("\nFeature engineering validation passed.")

FEATURE ENGINEERING COMPLETE
Original hourly rows : 719
Modeling rows        : 717
Total columns        : 24
Missing cells        : 0

Modeling time range:
Start: 2013-08-12 15:00:00+00:00
End  : 2013-09-11 11:00:00+00:00

Target statistics:


,target
count,717.000000
mean,38.235005
std,19.748840
min,1.335226
25%,21.774321
50%,36.274521
75%,55.980151
max,86.277015



Feature verification:


,time_bucket,estimated_cost_index,cost_lag_1,cpu_change,target_next_hour_cost
0,2013-08-12 15:00:00+00:00,38.996738,24.397998,2.617843,55.498785
1,2013-08-12 16:00:00+00:00,55.498785,38.996738,3.598211,56.427959
2,2013-08-12 17:00:00+00:00,56.427959,55.498785,-0.001196,54.375590
3,2013-08-12 18:00:00+00:00,54.375590,56.427959,-0.124158,54.391908
4,2013-08-12 19:00:00+00:00,54.391908,54.375590,0.069134,58.045948



Feature engineering validation passed.


In [ ]:
# ============================================================
# CELL 5 — FEATURE SELECTION AND CHRONOLOGICAL SPLIT
# ============================================================

FINOPS_FEATURE_COLUMNS = [
    "cpu_mean",
    "memory_mean_gb",
    "disk_activity_kbps",
    "network_activity_kbps",
    "active_vms",
    "resource_cost_index",
    "estimated_cost_index",
    "hour",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "cpu_change",
    "memory_change",
    "disk_change",
    "network_change",
    "active_vms_change",
    "resource_cost_change",
    "cost_lag_1"
]

FINOPS_TARGET_COLUMN = "target_next_hour_cost"

# Complete feature matrix and target
X_finops = finops_modeling_df[
    FINOPS_FEATURE_COLUMNS
].copy()

y_finops = finops_modeling_df[
    FINOPS_TARGET_COLUMN
].copy()

time_finops = finops_modeling_df[
    "time_bucket"
].copy()

# ------------------------------------------------------------
# Chronological 70% / 15% / 15% split
# ------------------------------------------------------------

total_samples = len(X_finops)

train_end = int(total_samples * 0.70)
validation_end = train_end + int(total_samples * 0.15)

X_train_finops = X_finops.iloc[:train_end].copy()
y_train_finops = y_finops.iloc[:train_end].copy()
time_train_finops = time_finops.iloc[:train_end].copy()

X_val_finops = X_finops.iloc[
    train_end:validation_end
].copy()

y_val_finops = y_finops.iloc[
    train_end:validation_end
].copy()

time_val_finops = time_finops.iloc[
    train_end:validation_end
].copy()

X_test_finops = X_finops.iloc[
    validation_end:
].copy()

y_test_finops = y_finops.iloc[
    validation_end:
].copy()

time_test_finops = time_finops.iloc[
    validation_end:
].copy()

# ------------------------------------------------------------
# Display split information
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Split": [
        "Training",
        "Validation",
        "Test"
    ],
    "Samples": [
        len(X_train_finops),
        len(X_val_finops),
        len(X_test_finops)
    ],
    "Feature start": [
        time_train_finops.iloc[0],
        time_val_finops.iloc[0],
        time_test_finops.iloc[0]
    ],
    "Feature end": [
        time_train_finops.iloc[-1],
        time_val_finops.iloc[-1],
        time_test_finops.iloc[-1]
    ]
})

print("FINOPS CHRONOLOGICAL DATA SPLIT")
print("=" * 65)

print("Total samples :", total_samples)
print("Feature count :", len(FINOPS_FEATURE_COLUMNS))

display(split_summary)

print("\nShapes:")
print("Training   :", X_train_finops.shape, y_train_finops.shape)
print("Validation :", X_val_finops.shape, y_val_finops.shape)
print("Test       :", X_test_finops.shape, y_test_finops.shape)

print("\nTarget means:")
print(f"Training   : {y_train_finops.mean():.4f}")
print(f"Validation : {y_val_finops.mean():.4f}")
print(f"Test       : {y_test_finops.mean():.4f}")

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert X_train_finops.shape == (501, 22)
assert X_val_finops.shape == (107, 22)
assert X_test_finops.shape == (109, 22)

assert len(X_train_finops) + len(X_val_finops) + len(X_test_finops) == 717

assert time_train_finops.max() < time_val_finops.min()
assert time_val_finops.max() < time_test_finops.min()

assert X_train_finops.columns.tolist() == FINOPS_FEATURE_COLUMNS
assert X_val_finops.columns.tolist() == FINOPS_FEATURE_COLUMNS
assert X_test_finops.columns.tolist() == FINOPS_FEATURE_COLUMNS

print("\nChronological split validation passed.")
print("No random shuffling was used.")

FINOPS CHRONOLOGICAL DATA SPLIT
Total samples : 717
Feature count : 22


,Split,Samples,Feature start,Feature end
0,Training,501,2013-08-12 15:00:00+00:00,2013-09-02 11:00:00+00:00
1,Validation,107,2013-09-02 12:00:00+00:00,2013-09-06 22:00:00+00:00
2,Test,109,2013-09-06 23:00:00+00:00,2013-09-11 11:00:00+00:00



Shapes:
Training   : (501, 22) (501,)
Validation : (107, 22) (107,)
Test       : (109, 22) (109,)

Target means:
Training   : 41.6597
Validation : 38.6021
Test       : 22.1335

Chronological split validation passed.
No random shuffling was used.


In [ ]:
# ============================================================
# CELL 6 — NAIVE PERSISTENCE BASELINE
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Current-hour cost becomes the next-hour prediction
naive_val_predictions_finops = (
    X_val_finops["estimated_cost_index"].to_numpy()
)

naive_test_predictions_finops = (
    X_test_finops["estimated_cost_index"].to_numpy()
)

# ------------------------------------------------------------
# Validation metrics
# ------------------------------------------------------------

naive_val_mae_finops = mean_absolute_error(
    y_val_finops,
    naive_val_predictions_finops
)

naive_val_rmse_finops = np.sqrt(
    mean_squared_error(
        y_val_finops,
        naive_val_predictions_finops
    )
)

naive_val_r2_finops = r2_score(
    y_val_finops,
    naive_val_predictions_finops
)

# ------------------------------------------------------------
# Test metrics
# ------------------------------------------------------------

naive_test_mae_finops = mean_absolute_error(
    y_test_finops,
    naive_test_predictions_finops
)

naive_test_rmse_finops = np.sqrt(
    mean_squared_error(
        y_test_finops,
        naive_test_predictions_finops
    )
)

naive_test_r2_finops = r2_score(
    y_test_finops,
    naive_test_predictions_finops
)

baseline_results_finops = pd.DataFrame({
    "Split": ["Validation", "Test"],
    "MAE": [
        naive_val_mae_finops,
        naive_test_mae_finops
    ],
    "RMSE": [
        naive_val_rmse_finops,
        naive_test_rmse_finops
    ],
    "R2": [
        naive_val_r2_finops,
        naive_test_r2_finops
    ]
})

print("NAIVE PERSISTENCE BASELINE")
print("=" * 55)

display(
    baseline_results_finops.round(4)
)

print("Test prediction rule:")
print("Current-hour cost → next-hour predicted cost")

print("\nFirst 5 test predictions:")
display(
    pd.DataFrame({
        "feature_time": time_test_finops.iloc[:5].to_numpy(),
        "current_hour_cost": (
            X_test_finops["estimated_cost_index"]
            .iloc[:5]
            .to_numpy()
        ),
        "actual_next_hour_cost": (
            y_test_finops.iloc[:5].to_numpy()
        ),
        "naive_prediction": (
            naive_test_predictions_finops[:5]
        )
    })
)

# Confirm that this is the expected full FinOps test set
assert len(naive_test_predictions_finops) == 109
assert len(y_test_finops) == 109

print("\nBaseline evaluation completed using 109 test observations.")

NAIVE PERSISTENCE BASELINE


,Split,MAE,RMSE,R2
0,Validation,5.8452,8.9171,0.7677
1,Test,7.8828,10.8645,0.1178


Test prediction rule:
Current-hour cost → next-hour predicted cost

First 5 test predictions:


,feature_time,current_hour_cost,actual_next_hour_cost,naive_prediction
0,2013-09-06 23:00:00+00:00,21.629732,32.037608,21.629732
1,2013-09-07 00:00:00+00:00,32.037608,40.416795,32.037608
2,2013-09-07 01:00:00+00:00,40.416795,19.763919,40.416795
3,2013-09-07 02:00:00+00:00,19.763919,30.830895,19.763919
4,2013-09-07 03:00:00+00:00,30.830895,22.854520,30.830895



Baseline evaluation completed using 109 test observations.


In [ ]:
# ============================================================
# CELL 7 — INSTALL AND VERIFY MODEL LIBRARIES
# ============================================================

%pip install -q xgboost lightgbm catboost mlflow

import sklearn
import xgboost
import lightgbm
import catboost
import mlflow

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

library_versions = pd.DataFrame({
    "Library": [
        "scikit-learn",
        "XGBoost",
        "LightGBM",
        "CatBoost",
        "MLflow"
    ],
    "Version": [
        sklearn.__version__,
        xgboost.__version__,
        lightgbm.__version__,
        catboost.__version__,
        mlflow.__version__
    ]
})

print("MODEL LIBRARIES READY")
print("=" * 50)

display(library_versions)

print("Required model classes:")
print("XGBoost :", XGBRegressor)
print("LightGBM:", LGBMRegressor)
print("CatBoost:", CatBoostRegressor)

# Prevent accidental use of classifier classes
assert "Regressor" in XGBRegressor.__name__
assert "Regressor" in LGBMRegressor.__name__
assert "Regressor" in CatBoostRegressor.__name__

print("\nRegression-library verification passed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9

,Library,Version
0,scikit-learn,1.6.1
1,XGBoost,3.4.1
2,LightGBM,4.6.0
3,CatBoost,1.2.10
4,MLflow,3.15.1


Required model classes:
XGBoost : <class 'xgboost.sklearn.XGBRegressor'>
LightGBM: <class 'lightgbm.sklearn.LGBMRegressor'>
CatBoost: <class 'catboost.core.CatBoostRegressor'>

Regression-library verification passed.


In [ ]:
# ============================================================
# CELL 8 — XGBOOST REGRESSION MODEL
# ============================================================

from xgboost import XGBRegressor

# Reusable evaluation function
def calculate_regression_metrics_finops(
    actual,
    predicted
):
    return {
        "MAE": mean_absolute_error(
            actual,
            predicted
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual,
                predicted
            )
        ),
        "R2": r2_score(
            actual,
            predicted
        )
    }


# ------------------------------------------------------------
# Train genuine FinOps XGBoost regressor
# ------------------------------------------------------------

xgb_finops = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_finops.fit(
    X_train_finops,
    y_train_finops
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

xgb_val_predictions_finops = xgb_finops.predict(
    X_val_finops
)

xgb_test_predictions_finops = xgb_finops.predict(
    X_test_finops
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

xgb_val_metrics_finops = (
    calculate_regression_metrics_finops(
        y_val_finops,
        xgb_val_predictions_finops
    )
)

xgb_test_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        xgb_test_predictions_finops
    )
)

xgb_results_finops = pd.DataFrame([
    {
        "Model": "XGBoost",
        "Split": "Validation",
        **xgb_val_metrics_finops
    },
    {
        "Model": "XGBoost",
        "Split": "Test",
        **xgb_test_metrics_finops
    }
])

print("XGBOOST FINOPS REGRESSION RESULTS")
print("=" * 60)

print("Model type:", type(xgb_finops))
print("Training samples:", len(X_train_finops))
print("Features:", X_train_finops.shape[1])

display(
    xgb_results_finops.round(4)
)

# ------------------------------------------------------------
# Compare against clean production baseline
# ------------------------------------------------------------

xgb_mae_change_pct_finops = (
    (
        naive_test_mae_finops
        - xgb_test_metrics_finops["MAE"]
    )
    / naive_test_mae_finops
    * 100
)

print("TEST PROMOTION CHECK")
print("-" * 60)

print(
    f"Naive baseline MAE: "
    f"{naive_test_mae_finops:.4f}"
)

print(
    f"XGBoost test MAE : "
    f"{xgb_test_metrics_finops['MAE']:.4f}"
)

print(
    f"MAE improvement  : "
    f"{xgb_mae_change_pct_finops:.2f}%"
)

if xgb_test_metrics_finops["MAE"] < naive_test_mae_finops:
    xgb_promotion_decision_finops = "accepted"
    print("\nDecision: XGBoost beats the baseline.")
else:
    xgb_promotion_decision_finops = "rejected"
    print("\nDecision: XGBoost does not beat the baseline.")

# Prevent classifier contamination
assert isinstance(xgb_finops, XGBRegressor)
assert len(xgb_test_predictions_finops) == 109

print("\nXGBoost regression evaluation completed.")

XGBOOST FINOPS REGRESSION RESULTS
Model type: <class 'xgboost.sklearn.XGBRegressor'>
Training samples: 501
Features: 22


,Model,Split,MAE,RMSE,R2
0,XGBoost,Validation,5.7429,8.0521,0.8106
1,XGBoost,Test,10.0239,12.0580,-0.0867


TEST PROMOTION CHECK
------------------------------------------------------------
Naive baseline MAE: 7.8828
XGBoost test MAE : 10.0239
MAE improvement  : -27.16%

Decision: XGBoost does not beat the baseline.

XGBoost regression evaluation completed.


In [ ]:
# ============================================================
# CELL 9 — LIGHTGBM REGRESSION MODEL
# ============================================================

from lightgbm import LGBMRegressor

lgbm_finops = LGBMRegressor(
    n_estimators=300,
    max_depth=4,
    num_leaves=15,
    learning_rate=0.05,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_finops.fit(
    X_train_finops,
    y_train_finops
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

lgbm_val_predictions_finops = lgbm_finops.predict(
    X_val_finops
)

lgbm_test_predictions_finops = lgbm_finops.predict(
    X_test_finops
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

lgbm_val_metrics_finops = (
    calculate_regression_metrics_finops(
        y_val_finops,
        lgbm_val_predictions_finops
    )
)

lgbm_test_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        lgbm_test_predictions_finops
    )
)

lgbm_results_finops = pd.DataFrame([
    {
        "Model": "LightGBM",
        "Split": "Validation",
        **lgbm_val_metrics_finops
    },
    {
        "Model": "LightGBM",
        "Split": "Test",
        **lgbm_test_metrics_finops
    }
])

print("LIGHTGBM FINOPS REGRESSION RESULTS")
print("=" * 60)

print("Model type:", type(lgbm_finops))
print("Training samples:", len(X_train_finops))
print("Features:", X_train_finops.shape[1])

display(
    lgbm_results_finops.round(4)
)

# ------------------------------------------------------------
# Test-set comparison
# ------------------------------------------------------------

lgbm_test_comparison_finops = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "XGBoost",
        "LightGBM"
    ],
    "Test MAE": [
        naive_test_mae_finops,
        xgb_test_metrics_finops["MAE"],
        lgbm_test_metrics_finops["MAE"]
    ],
    "Test RMSE": [
        naive_test_rmse_finops,
        xgb_test_metrics_finops["RMSE"],
        lgbm_test_metrics_finops["RMSE"]
    ],
    "Test R2": [
        naive_test_r2_finops,
        xgb_test_metrics_finops["R2"],
        lgbm_test_metrics_finops["R2"]
    ]
}).sort_values(
    "Test MAE",
    ascending=True
).reset_index(drop=True)

print("TEST-SET COMPARISON")
print("-" * 60)

display(
    lgbm_test_comparison_finops.round(4)
)

lgbm_mae_improvement_finops = (
    (
        naive_test_mae_finops
        - lgbm_test_metrics_finops["MAE"]
    )
    / naive_test_mae_finops
    * 100
)

print(
    f"LightGBM improvement over baseline: "
    f"{lgbm_mae_improvement_finops:.2f}%"
)

if lgbm_test_metrics_finops["MAE"] < naive_test_mae_finops:
    lgbm_promotion_decision_finops = "accepted"
    print("Decision: LightGBM beats the baseline.")
else:
    lgbm_promotion_decision_finops = "rejected"
    print("Decision: LightGBM does not beat the baseline.")

assert isinstance(lgbm_finops, LGBMRegressor)
assert len(lgbm_test_predictions_finops) == 109

print("\nLightGBM regression evaluation completed.")

LIGHTGBM FINOPS REGRESSION RESULTS
Model type: <class 'lightgbm.sklearn.LGBMRegressor'>
Training samples: 501
Features: 22


,Model,Split,MAE,RMSE,R2
0,LightGBM,Validation,6.2224,8.5528,0.7863
1,LightGBM,Test,9.2050,11.5957,-0.0050


TEST-SET COMPARISON
------------------------------------------------------------


,Model,Test MAE,Test RMSE,Test R2
0,Naive Baseline,7.8828,10.8645,0.1178
1,LightGBM,9.2050,11.5957,-0.0050
2,XGBoost,10.0239,12.0580,-0.0867


LightGBM improvement over baseline: -16.77%
Decision: LightGBM does not beat the baseline.

LightGBM regression evaluation completed.


In [ ]:
# ============================================================
# CELL 10 — CATBOOST REGRESSION MODEL
# ============================================================

from catboost import CatBoostRegressor

catboost_finops = CatBoostRegressor(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

catboost_finops.fit(
    X_train_finops,
    y_train_finops
)

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

catboost_val_predictions_finops = catboost_finops.predict(
    X_val_finops
)

catboost_test_predictions_finops = catboost_finops.predict(
    X_test_finops
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

catboost_val_metrics_finops = (
    calculate_regression_metrics_finops(
        y_val_finops,
        catboost_val_predictions_finops
    )
)

catboost_test_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        catboost_test_predictions_finops
    )
)

catboost_results_finops = pd.DataFrame([
    {
        "Model": "CatBoost",
        "Split": "Validation",
        **catboost_val_metrics_finops
    },
    {
        "Model": "CatBoost",
        "Split": "Test",
        **catboost_test_metrics_finops
    }
])

print("CATBOOST FINOPS REGRESSION RESULTS")
print("=" * 60)

print("Model type:", type(catboost_finops))
print("Training samples:", len(X_train_finops))
print("Features:", X_train_finops.shape[1])

display(
    catboost_results_finops.round(4)
)

# ------------------------------------------------------------
# Complete test comparison
# ------------------------------------------------------------

all_models_test_comparison_finops = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "Test MAE": [
        naive_test_mae_finops,
        xgb_test_metrics_finops["MAE"],
        lgbm_test_metrics_finops["MAE"],
        catboost_test_metrics_finops["MAE"]
    ],
    "Test RMSE": [
        naive_test_rmse_finops,
        xgb_test_metrics_finops["RMSE"],
        lgbm_test_metrics_finops["RMSE"],
        catboost_test_metrics_finops["RMSE"]
    ],
    "Test R2": [
        naive_test_r2_finops,
        xgb_test_metrics_finops["R2"],
        lgbm_test_metrics_finops["R2"],
        catboost_test_metrics_finops["R2"]
    ]
}).sort_values(
    "Test MAE",
    ascending=True
).reset_index(drop=True)

print("COMPLETE TEST-SET COMPARISON")
print("-" * 60)

display(
    all_models_test_comparison_finops.round(4)
)

catboost_mae_improvement_finops = (
    (
        naive_test_mae_finops
        - catboost_test_metrics_finops["MAE"]
    )
    / naive_test_mae_finops
    * 100
)

print(
    f"CatBoost improvement over baseline: "
    f"{catboost_mae_improvement_finops:.2f}%"
)

if catboost_test_metrics_finops["MAE"] < naive_test_mae_finops:
    catboost_promotion_decision_finops = "accepted"
    print("Decision: CatBoost beats the baseline.")
else:
    catboost_promotion_decision_finops = "rejected"
    print("Decision: CatBoost does not beat the baseline.")

assert isinstance(catboost_finops, CatBoostRegressor)
assert len(catboost_test_predictions_finops) == 109

print("\nCatBoost regression evaluation completed.")


CATBOOST FINOPS REGRESSION RESULTS
Model type: <class 'catboost.core.CatBoostRegressor'>
Training samples: 501
Features: 22


,Model,Split,MAE,RMSE,R2
0,CatBoost,Validation,5.6986,7.9919,0.8134
1,CatBoost,Test,8.5622,10.6691,0.1492


COMPLETE TEST-SET COMPARISON
------------------------------------------------------------


,Model,Test MAE,Test RMSE,Test R2
0,Naive Baseline,7.8828,10.8645,0.1178
1,CatBoost,8.5622,10.6691,0.1492
2,LightGBM,9.2050,11.5957,-0.0050
3,XGBoost,10.0239,12.0580,-0.0867


CatBoost improvement over baseline: -8.62%
Decision: CatBoost does not beat the baseline.

CatBoost regression evaluation completed.


In [ ]:
# ============================================================
# CELL 11 — VALIDATION VS TEST GENERALIZATION ANALYSIS
# ============================================================

generalization_analysis_finops = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "Validation MAE": [
        naive_val_mae_finops,
        xgb_val_metrics_finops["MAE"],
        lgbm_val_metrics_finops["MAE"],
        catboost_val_metrics_finops["MAE"]
    ],
    "Test MAE": [
        naive_test_mae_finops,
        xgb_test_metrics_finops["MAE"],
        lgbm_test_metrics_finops["MAE"],
        catboost_test_metrics_finops["MAE"]
    ]
})

generalization_analysis_finops["MAE Increase"] = (
    generalization_analysis_finops["Test MAE"]
    - generalization_analysis_finops["Validation MAE"]
)

generalization_analysis_finops["MAE Increase %"] = (
    generalization_analysis_finops["MAE Increase"]
    / generalization_analysis_finops["Validation MAE"]
    * 100
)

generalization_analysis_finops["Promotion Decision"] = np.where(
    generalization_analysis_finops["Test MAE"]
    <= naive_test_mae_finops,
    "Production / Accepted",
    "Rejected"
)

# The baseline is the current production reference
generalization_analysis_finops.loc[
    generalization_analysis_finops["Model"] == "Naive Baseline",
    "Promotion Decision"
] = "Production"

print("VALIDATION → TEST GENERALIZATION ANALYSIS")
print("=" * 75)

display(
    generalization_analysis_finops.round(4)
)

# ------------------------------------------------------------
# Target-distribution comparison
# ------------------------------------------------------------

target_distribution_finops = pd.DataFrame({
    "Split": [
        "Training",
        "Validation",
        "Test"
    ],
    "Samples": [
        len(y_train_finops),
        len(y_val_finops),
        len(y_test_finops)
    ],
    "Target Mean": [
        y_train_finops.mean(),
        y_val_finops.mean(),
        y_test_finops.mean()
    ],
    "Target Std": [
        y_train_finops.std(),
        y_val_finops.std(),
        y_test_finops.std()
    ],
    "Target Minimum": [
        y_train_finops.min(),
        y_val_finops.min(),
        y_test_finops.min()
    ],
    "Target Maximum": [
        y_train_finops.max(),
        y_val_finops.max(),
        y_test_finops.max()
    ]
})

print("TARGET DISTRIBUTION BY SPLIT")
print("-" * 75)

display(
    target_distribution_finops.round(4)
)

pretest_samples_finops = (
    len(X_train_finops)
    + len(X_val_finops)
)

print("RETRAINING ASSESSMENT")
print("-" * 75)

print(
    "Training + validation samples available:",
    pretest_samples_finops
)

print(
    f"Training target mean  : "
    f"{y_train_finops.mean():.4f}"
)

print(
    f"Validation target mean: "
    f"{y_val_finops.mean():.4f}"
)

print(
    f"Test target mean      : "
    f"{y_test_finops.mean():.4f}"
)

print(
    "\nConclusion: temporal degradation is present. "
    "A drift-triggered retraining candidate is justified, "
    "but it must still beat the untouched test baseline "
    "before promotion."
)

assert pretest_samples_finops == 608

VALIDATION → TEST GENERALIZATION ANALYSIS


,Model,Validation MAE,Test MAE,MAE Increase,MAE Increase %,Promotion Decision
0,Naive Baseline,5.8452,7.8828,2.0377,34.8609,Production
1,XGBoost,5.7429,10.0239,4.2810,74.5448,Rejected
2,LightGBM,6.2224,9.2050,2.9826,47.9337,Rejected
3,CatBoost,5.6986,8.5622,2.8636,50.2512,Rejected


TARGET DISTRIBUTION BY SPLIT
---------------------------------------------------------------------------


,Split,Samples,Target Mean,Target Std,Target Minimum,Target Maximum
0,Training,501,41.6597,19.6924,1.3352,86.2770
1,Validation,107,38.6021,18.5879,9.4102,75.3687
2,Test,109,22.1335,11.6202,4.6349,58.3604


RETRAINING ASSESSMENT
---------------------------------------------------------------------------
Training + validation samples available: 608
Training target mean  : 41.6597
Validation target mean: 38.6021
Test target mean      : 22.1335

Conclusion: temporal degradation is present. A drift-triggered retraining candidate is justified, but it must still beat the untouched test baseline before promotion.


In [ ]:
# ============================================================
# CELL 12 — DRIFT-TRIGGERED XGBOOST RETRAINING
# ============================================================

# Combine chronologically earlier training and validation data
X_retrain_finops = pd.concat(
    [
        X_train_finops,
        X_val_finops
    ],
    axis=0,
    ignore_index=True
)

y_retrain_finops = pd.concat(
    [
        y_train_finops,
        y_val_finops
    ],
    axis=0,
    ignore_index=True
)

time_retrain_finops = pd.concat(
    [
        time_train_finops,
        time_val_finops
    ],
    axis=0,
    ignore_index=True
)

print("DRIFT RETRAINING DATA")
print("=" * 60)

print("Retraining features:", X_retrain_finops.shape)
print("Retraining target  :", y_retrain_finops.shape)
print("Retraining start   :", time_retrain_finops.iloc[0])
print("Retraining end     :", time_retrain_finops.iloc[-1])
print("Test start         :", time_test_finops.iloc[0])

# ------------------------------------------------------------
# Train the drift-retrained candidate
# ------------------------------------------------------------

retrained_xgb_finops = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

retrained_xgb_finops.fit(
    X_retrain_finops,
    y_retrain_finops
)

retrained_xgb_test_predictions_finops = (
    retrained_xgb_finops.predict(
        X_test_finops
    )
)

retrained_xgb_test_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        retrained_xgb_test_predictions_finops
    )
)

# ------------------------------------------------------------
# Compare retrained model
# ------------------------------------------------------------

retraining_comparison_finops = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Original XGBoost",
        "Drift-Retrained XGBoost"
    ],
    "Training Samples": [
        0,
        len(X_train_finops),
        len(X_retrain_finops)
    ],
    "Test MAE": [
        naive_test_mae_finops,
        xgb_test_metrics_finops["MAE"],
        retrained_xgb_test_metrics_finops["MAE"]
    ],
    "Test RMSE": [
        naive_test_rmse_finops,
        xgb_test_metrics_finops["RMSE"],
        retrained_xgb_test_metrics_finops["RMSE"]
    ],
    "Test R2": [
        naive_test_r2_finops,
        xgb_test_metrics_finops["R2"],
        retrained_xgb_test_metrics_finops["R2"]
    ]
}).sort_values(
    "Test MAE",
    ascending=True
).reset_index(drop=True)

print("\nDRIFT-RETRAINED XGBOOST RESULTS")
print("=" * 60)

print("Model type:", type(retrained_xgb_finops))

display(
    retraining_comparison_finops.round(4)
)

# Improvement relative to original XGBoost
retrained_improvement_vs_original_finops = (
    (
        xgb_test_metrics_finops["MAE"]
        - retrained_xgb_test_metrics_finops["MAE"]
    )
    / xgb_test_metrics_finops["MAE"]
    * 100
)

# Improvement relative to production baseline
retrained_improvement_vs_baseline_finops = (
    (
        naive_test_mae_finops
        - retrained_xgb_test_metrics_finops["MAE"]
    )
    / naive_test_mae_finops
    * 100
)

print(
    "Improvement over original XGBoost: "
    f"{retrained_improvement_vs_original_finops:.2f}%"
)

print(
    "Improvement over production baseline: "
    f"{retrained_improvement_vs_baseline_finops:.2f}%"
)

# ------------------------------------------------------------
# Promotion gate
# ------------------------------------------------------------

if (
    retrained_xgb_test_metrics_finops["MAE"]
    < naive_test_mae_finops
):
    retrained_xgb_promotion_decision_finops = "accepted"
    retrained_xgb_deployment_status_finops = "candidate_for_promotion"

    print(
        "\nPromotion decision: ACCEPTED — candidate "
        "beats the production baseline."
    )

else:
    retrained_xgb_promotion_decision_finops = "rejected"
    retrained_xgb_deployment_status_finops = "rejected"

    print(
        "\nPromotion decision: REJECTED — candidate "
        "does not beat the production baseline."
    )

assert X_retrain_finops.shape == (608, 22)
assert isinstance(retrained_xgb_finops, XGBRegressor)
assert len(retrained_xgb_test_predictions_finops) == 109

print("\nDrift-retraining evaluation completed.")

DRIFT RETRAINING DATA
Retraining features: (608, 22)
Retraining target  : (608,)
Retraining start   : 2013-08-12 15:00:00+00:00
Retraining end     : 2013-09-06 22:00:00+00:00
Test start         : 2013-09-06 23:00:00+00:00

DRIFT-RETRAINED XGBOOST RESULTS
Model type: <class 'xgboost.sklearn.XGBRegressor'>


,Model,Training Samples,Test MAE,Test RMSE,Test R2
0,Naive Baseline,0,7.8828,10.8645,0.1178
1,Drift-Retrained XGBoost,608,9.1266,10.7302,0.1394
2,Original XGBoost,501,10.0239,12.0580,-0.0867


Improvement over original XGBoost: 8.95%
Improvement over production baseline: -15.78%

Promotion decision: REJECTED — candidate does not beat the production baseline.

Drift-retraining evaluation completed.


In [ ]:
# ============================================================
# CELL 13 — AUTHORITATIVE MODEL DECISION TABLE
# ============================================================

final_model_decisions_finops = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "XGBoost",
        "LightGBM",
        "CatBoost",
        "Drift-Retrained XGBoost"
    ],
    "Training Samples": [
        0,
        len(X_train_finops),
        len(X_train_finops),
        len(X_train_finops),
        len(X_retrain_finops)
    ],
    "Validation MAE": [
        naive_val_mae_finops,
        xgb_val_metrics_finops["MAE"],
        lgbm_val_metrics_finops["MAE"],
        catboost_val_metrics_finops["MAE"],
        np.nan
    ],
    "Test MAE": [
        naive_test_mae_finops,
        xgb_test_metrics_finops["MAE"],
        lgbm_test_metrics_finops["MAE"],
        catboost_test_metrics_finops["MAE"],
        retrained_xgb_test_metrics_finops["MAE"]
    ],
    "Test RMSE": [
        naive_test_rmse_finops,
        xgb_test_metrics_finops["RMSE"],
        lgbm_test_metrics_finops["RMSE"],
        catboost_test_metrics_finops["RMSE"],
        retrained_xgb_test_metrics_finops["RMSE"]
    ],
    "Test R2": [
        naive_test_r2_finops,
        xgb_test_metrics_finops["R2"],
        lgbm_test_metrics_finops["R2"],
        catboost_test_metrics_finops["R2"],
        retrained_xgb_test_metrics_finops["R2"]
    ],
    "Decision": [
        "Production",
        xgb_promotion_decision_finops,
        lgbm_promotion_decision_finops,
        catboost_promotion_decision_finops,
        retrained_xgb_promotion_decision_finops
    ]
})

final_model_decisions_finops[
    "MAE Improvement vs Baseline %"
] = (
    (
        naive_test_mae_finops
        - final_model_decisions_finops["Test MAE"]
    )
    / naive_test_mae_finops
    * 100
)

final_model_decisions_finops = (
    final_model_decisions_finops
    .sort_values("Test MAE", ascending=True)
    .reset_index(drop=True)
)

final_model_decisions_finops.insert(
    0,
    "MAE Rank",
    range(1, len(final_model_decisions_finops) + 1)
)

print("FINAL CLEAN FINOPS MODEL DECISIONS")
print("=" * 90)

display(
    final_model_decisions_finops.round(4)
)

best_overall_model_finops = (
    final_model_decisions_finops.iloc[0]["Model"]
)

best_learned_model_finops = (
    final_model_decisions_finops[
        final_model_decisions_finops["Model"]
        != "Naive Baseline"
    ]
    .sort_values("Test MAE")
    .iloc[0]["Model"]
)

print("FINAL LIFECYCLE DECISION")
print("-" * 90)

print(
    "Primary promotion metric : Test MAE"
)

print(
    "Best overall model       :",
    best_overall_model_finops
)

print(
    "Best learned model       :",
    best_learned_model_finops
)

print(
    "Production model retained: Naive Baseline"
)

print(
    "Drift retraining outcome : Candidate rejected"
)

print(
    "\nReason: no trained model achieved a lower "
    "test MAE than the production baseline."
)

# Strict lifecycle checks
assert best_overall_model_finops == "Naive Baseline"
assert best_learned_model_finops == "CatBoost"
assert retrained_xgb_promotion_decision_finops == "rejected"

print("\nFinal decision table validation passed.")

FINAL CLEAN FINOPS MODEL DECISIONS


,MAE Rank,Model,Training Samples,Validation MAE,Test MAE,Test RMSE,Test R2,Decision,MAE Improvement vs Baseline %
0,1,Naive Baseline,0,5.8452,7.8828,10.8645,0.1178,Production,0.0000
1,2,CatBoost,501,5.6986,8.5622,10.6691,0.1492,rejected,-8.6182
2,3,Drift-Retrained XGBoost,608,NaN,9.1266,10.7302,0.1394,rejected,-15.7781
3,4,LightGBM,501,6.2224,9.2050,11.5957,-0.0050,rejected,-16.7729
4,5,XGBoost,501,5.7429,10.0239,12.0580,-0.0867,rejected,-27.1611


FINAL LIFECYCLE DECISION
------------------------------------------------------------------------------------------
Primary promotion metric : Test MAE
Best overall model       : Naive Baseline
Best learned model       : CatBoost
Production model retained: Naive Baseline
Drift retraining outcome : Candidate rejected

Reason: no trained model achieved a lower test MAE than the production baseline.

Final decision table validation passed.


In [ ]:
# ============================================================
# CELL 14 — CLEAN PERSISTENT MLFLOW EXPERIMENT
# ============================================================

from pathlib import Path
import mlflow

MLFLOW_ROOT_FINOPS = Path(
    "/content/drive/MyDrive/finops_mlflow"
)

MLFLOW_ROOT_FINOPS.mkdir(
    parents=True,
    exist_ok=True
)

# Persistent SQLite metadata backend
MLFLOW_DB_FINOPS = (
    MLFLOW_ROOT_FINOPS / "mlflow.db"
)

MLFLOW_TRACKING_URI_FINOPS = (
    f"sqlite:///{MLFLOW_DB_FINOPS}"
)

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI_FINOPS
)

# Separate artifacts from the contaminated experiment
CLEAN_ARTIFACT_DIR_FINOPS = (
    MLFLOW_ROOT_FINOPS / "artifacts_clean_v1"
)

CLEAN_ARTIFACT_DIR_FINOPS.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_ARTIFACT_URI_FINOPS = (
    CLEAN_ARTIFACT_DIR_FINOPS.as_uri()
)

CLEAN_EXPERIMENT_NAME_FINOPS = (
    "finops-cloud-cost-forecasting-clean-v1"
)

# Do not create/select an experiment while another run is active
if mlflow.active_run() is not None:
    raise RuntimeError(
        "An MLflow run is already active. End it before continuing."
    )

existing_clean_experiment_finops = (
    mlflow.get_experiment_by_name(
        CLEAN_EXPERIMENT_NAME_FINOPS
    )
)

if existing_clean_experiment_finops is None:
    clean_experiment_id_finops = (
        mlflow.create_experiment(
            name=CLEAN_EXPERIMENT_NAME_FINOPS,
            artifact_location=CLEAN_ARTIFACT_URI_FINOPS
        )
    )

    print("New clean MLflow experiment created.")

else:
    clean_experiment_id_finops = (
        existing_clean_experiment_finops.experiment_id
    )

    print("Existing clean MLflow experiment found.")

mlflow.set_experiment(
    CLEAN_EXPERIMENT_NAME_FINOPS
)

clean_experiment_finops = (
    mlflow.get_experiment_by_name(
        CLEAN_EXPERIMENT_NAME_FINOPS
    )
)

# Inspect current clean-experiment history
existing_clean_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ]
)

print("\nCLEAN MLFLOW CONFIGURATION")
print("=" * 75)

print(
    "Tracking URI      :",
    mlflow.get_tracking_uri()
)

print(
    "Experiment name   :",
    clean_experiment_finops.name
)

print(
    "Experiment ID     :",
    clean_experiment_finops.experiment_id
)

print(
    "Artifact location :",
    clean_experiment_finops.artifact_location
)

print(
    "Existing run count:",
    len(existing_clean_runs_finops)
)

# Strict isolation checks
assert (
    clean_experiment_finops.name
    == CLEAN_EXPERIMENT_NAME_FINOPS
)

assert (
    "artifacts_clean_v1"
    in clean_experiment_finops.artifact_location
)

if len(existing_clean_runs_finops) == 0:
    print(
        "\nClean experiment is empty and ready for logging."
    )
else:
    print(
        "\nWarning: this clean experiment already contains runs. "
        "Do not log duplicates until they are inspected."
    )

2026/08/24 04:43:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/24 04:43:53 INFO mlflow.store.db.utils: Updating database tables


New clean MLflow experiment created.

CLEAN MLFLOW CONFIGURATION
Tracking URI      : sqlite:////content/drive/MyDrive/finops_mlflow/mlflow.db
Experiment name   : finops-cloud-cost-forecasting-clean-v1
Experiment ID     : 1
Artifact location : file:///content/drive/MyDrive/finops_mlflow/artifacts_clean_v1
Existing run count: 0

Clean experiment is empty and ready for logging.


In [ ]:
# ============================================================
# REPLACEMENT CELL 15 — LOG TRUSTED NAIVE BASELINE
# ============================================================

from sklearn.linear_model import LinearRegression
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

mlflow.set_experiment(
    CLEAN_EXPERIMENT_NAME_FINOPS
)

clean_experiment_finops = (
    mlflow.get_experiment_by_name(
        CLEAN_EXPERIMENT_NAME_FINOPS
    )
)

client_finops = MlflowClient()

# ------------------------------------------------------------
# Soft-delete only failed attempts from the previous cell
# ------------------------------------------------------------

previous_baseline_attempts_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        "tags.mlflow.runName = "
        "'naive_baseline_clean'"
    )
)

deleted_failed_run_ids_finops = []

for _, failed_run in previous_baseline_attempts_finops.iterrows():

    if failed_run["status"] == "FAILED":

        client_finops.delete_run(
            failed_run["run_id"]
        )

        deleted_failed_run_ids_finops.append(
            failed_run["run_id"]
        )

print(
    "Soft-deleted failed baseline runs:",
    len(deleted_failed_run_ids_finops)
)

for run_id in deleted_failed_run_ids_finops:
    print("-", run_id)

# ------------------------------------------------------------
# Create a trusted standard sklearn identity model
# ------------------------------------------------------------

BASELINE_INPUT_COLUMNS_FINOPS = [
    "estimated_cost_index"
]

baseline_X_train_finops = (
    X_train_finops[
        BASELINE_INPUT_COLUMNS_FINOPS
    ]
    .astype("float64")
    .copy()
)

baseline_identity_target_finops = (
    X_train_finops[
        "estimated_cost_index"
    ]
    .astype("float64")
    .copy()
)

baseline_X_test_finops = (
    X_test_finops[
        BASELINE_INPUT_COLUMNS_FINOPS
    ]
    .astype("float64")
    .copy()
)

naive_baseline_model_finops = LinearRegression()

# Fit y = 1 × current cost + 0
naive_baseline_model_finops.fit(
    baseline_X_train_finops,
    baseline_identity_target_finops
)

baseline_model_test_predictions_finops = (
    naive_baseline_model_finops.predict(
        baseline_X_test_finops
    )
)

# Verify it implements the persistence rule
assert np.allclose(
    baseline_model_test_predictions_finops,
    naive_test_predictions_finops,
    atol=1e-10
)

print("\nBaseline model verification:")
print(
    "Coefficient:",
    naive_baseline_model_finops.coef_[0]
)
print(
    "Intercept  :",
    naive_baseline_model_finops.intercept_
)
print(
    "Model type :",
    type(naive_baseline_model_finops)
)

# ------------------------------------------------------------
# Check whether a successful run already exists
# ------------------------------------------------------------

remaining_baseline_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        "tags.mlflow.runName = "
        "'naive_baseline_clean'"
    )
)

finished_baseline_runs_finops = (
    remaining_baseline_runs_finops[
        remaining_baseline_runs_finops["status"]
        == "FINISHED"
    ]
)

if not finished_baseline_runs_finops.empty:

    clean_naive_run_id_finops = (
        finished_baseline_runs_finops
        .iloc[0]["run_id"]
    )

    print("\nSuccessful baseline run already exists.")
    print("Run ID:", clean_naive_run_id_finops)

else:

    baseline_input_example_finops = (
        baseline_X_train_finops
        .head(5)
        .copy()
    )

    baseline_signature_finops = infer_signature(
        baseline_input_example_finops,
        naive_baseline_model_finops.predict(
            baseline_input_example_finops
        )
    )

    with mlflow.start_run(
        run_name="naive_baseline_clean"
    ) as run:

        mlflow.log_params({
            "model_type": "persistence_baseline",
            "implementation": "sklearn_linear_identity_model",
            "prediction_rule": (
                "current_hour_cost_predicts_next_hour_cost"
            ),
            "forecast_horizon": "1_hour",
            "dataset_version": (
                "bitbrains_hourly_clean_719_rows"
            ),
            "modeling_rows": 717,
            "validation_samples": 107,
            "test_samples": 109,
            "input_feature_count": 1
        })

        mlflow.log_metrics({
            "validation_mae": naive_val_mae_finops,
            "validation_rmse": naive_val_rmse_finops,
            "validation_r2": naive_val_r2_finops,
            "test_mae": naive_test_mae_finops,
            "test_rmse": naive_test_rmse_finops,
            "test_r2": naive_test_r2_finops
        })

        mlflow.set_tags({
            "project": "FinOps Cloud Cost Forecasting",
            "model_family": "baseline",
            "dataset_status": "clean",
            "deployment_status": "production",
            "promotion_decision": "retained",
            "primary_promotion_metric": "test_mae"
        })

        mlflow.sklearn.log_model(
            sk_model=naive_baseline_model_finops,
            name="model",
            signature=baseline_signature_finops,
            input_example=baseline_input_example_finops
        )

        mlflow.log_dict(
            {
                "model_input_features": (
                    BASELINE_INPUT_COLUMNS_FINOPS
                ),
                "prediction_rule": (
                    "prediction(t+1) = "
                    "estimated_cost_index(t)"
                ),
                "dataset_rows": 719,
                "modeling_rows": 717,
                "split": {
                    "train": 501,
                    "validation": 107,
                    "test": 109
                }
            },
            "metadata/baseline_schema.json"
        )

        clean_naive_run_id_finops = (
            run.info.run_id
        )

    print("\nClean trusted baseline logged successfully.")
    print("Run ID:", clean_naive_run_id_finops)

print(
    "Test MAE:",
    round(naive_test_mae_finops, 4)
)

Soft-deleted failed baseline runs: 1
- 9de058d74f5743feab8e954e08983b7f

Baseline model verification:
Coefficient: 0.9999999999999999
Intercept  : 7.105427357601002e-15
Model type : <class 'sklearn.linear_model._base.LinearRegression'>

Clean trusted baseline logged successfully.
Run ID: 25b920c78baf4274a36b9cea73f39e23
Test MAE: 7.8828


In [ ]:
# ============================================================
# CELL 16 — VERIFY CLEAN MLFLOW BASELINE
# ============================================================

mlflow.set_experiment(
    CLEAN_EXPERIMENT_NAME_FINOPS
)

clean_experiment_finops = (
    mlflow.get_experiment_by_name(
        CLEAN_EXPERIMENT_NAME_FINOPS
    )
)

clean_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ]
)

columns_to_show_finops = [
    "run_id",
    "status",
    "tags.mlflow.runName",
    "metrics.validation_mae",
    "metrics.test_mae",
    "metrics.test_rmse",
    "metrics.test_r2",
    "tags.deployment_status",
    "tags.promotion_decision"
]

columns_to_show_finops = [
    column
    for column in columns_to_show_finops
    if column in clean_runs_finops.columns
]

print("CLEAN MLFLOW EXPERIMENT")
print("=" * 80)

display(
    clean_runs_finops[
        columns_to_show_finops
    ]
)

successful_baseline_finops = clean_runs_finops[
    (
        clean_runs_finops["tags.mlflow.runName"]
        == "naive_baseline_clean"
    )
    &
    (
        clean_runs_finops["status"]
        == "FINISHED"
    )
]

print("Visible active runs:", len(clean_runs_finops))
print(
    "Successful baseline runs:",
    len(successful_baseline_finops)
)

assert len(successful_baseline_finops) == 1

clean_naive_run_id_finops = (
    successful_baseline_finops.iloc[0]["run_id"]
)

print("\nClean baseline verification passed.")
print("Run ID:", clean_naive_run_id_finops)

CLEAN MLFLOW EXPERIMENT


,run_id,status,tags.mlflow.runName,metrics.validation_mae,metrics.test_mae,metrics.test_rmse,metrics.test_r2,tags.deployment_status,tags.promotion_decision
0,25b920c78baf4274a36b9cea73f39e23,FINISHED,naive_baseline_clean,5.845166,7.882846,10.864452,0.117757,production,retained


Visible active runs: 1
Successful baseline runs: 1

Clean baseline verification passed.
Run ID: 25b920c78baf4274a36b9cea73f39e23


In [ ]:
# ============================================================
# CELL 17 — LOG CLEAN XGBOOST TO MLFLOW
# ============================================================

from mlflow.models import infer_signature

mlflow.set_experiment(
    CLEAN_EXPERIMENT_NAME_FINOPS
)

RUN_NAME_XGB_FINOPS = "xgboost_clean"

existing_xgb_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        f"tags.mlflow.runName = "
        f"'{RUN_NAME_XGB_FINOPS}'"
    )
)

finished_xgb_runs_finops = (
    existing_xgb_runs_finops[
        existing_xgb_runs_finops["status"]
        == "FINISHED"
    ]
)

if not finished_xgb_runs_finops.empty:

    clean_xgb_run_id_finops = (
        finished_xgb_runs_finops
        .iloc[0]["run_id"]
    )

    print("Clean XGBoost run already exists.")
    print("Run ID:", clean_xgb_run_id_finops)

else:

    # Use float64 inputs for a robust MLflow schema
    xgb_input_example_finops = (
        X_train_finops
        .head(5)
        .astype("float64")
        .copy()
    )

    xgb_signature_finops = infer_signature(
        xgb_input_example_finops,
        xgb_finops.predict(
            xgb_input_example_finops
        )
    )

    xgb_feature_importance_finops = {
        feature: float(importance)
        for feature, importance in zip(
            FINOPS_FEATURE_COLUMNS,
            xgb_finops.feature_importances_
        )
    }

    with mlflow.start_run(
        run_name=RUN_NAME_XGB_FINOPS
    ) as run:

        mlflow.log_params({
            "model_type": "XGBRegressor",
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "forecast_horizon": "1_hour",
            "training_samples": 501,
            "validation_samples": 107,
            "test_samples": 109,
            "num_features": 22,
            "dataset_version": (
                "bitbrains_hourly_clean_719_rows"
            )
        })

        mlflow.log_metrics({
            "validation_mae": (
                xgb_val_metrics_finops["MAE"]
            ),
            "validation_rmse": (
                xgb_val_metrics_finops["RMSE"]
            ),
            "validation_r2": (
                xgb_val_metrics_finops["R2"]
            ),
            "test_mae": (
                xgb_test_metrics_finops["MAE"]
            ),
            "test_rmse": (
                xgb_test_metrics_finops["RMSE"]
            ),
            "test_r2": (
                xgb_test_metrics_finops["R2"]
            ),
            "production_baseline_mae": (
                naive_test_mae_finops
            ),
            "mae_improvement_vs_baseline_pct": (
                xgb_mae_change_pct_finops
            )
        })

        mlflow.set_tags({
            "project": "FinOps Cloud Cost Forecasting",
            "model_family": "gradient_boosting",
            "dataset_status": "clean",
            "deployment_status": "rejected",
            "promotion_decision": "rejected",
            "primary_promotion_metric": "test_mae",
            "rejection_reason": (
                "failed_to_beat_naive_baseline_on_test"
            )
        })

        xgb_logged_model_info_finops = (
            mlflow.xgboost.log_model(
                xgb_model=xgb_finops,
                name="model",
                signature=xgb_signature_finops,
                input_example=xgb_input_example_finops
            )
        )

        mlflow.log_dict(
            xgb_feature_importance_finops,
            "metadata/feature_importance.json"
        )

        clean_xgb_run_id_finops = (
            run.info.run_id
        )

    print("Clean XGBoost logged.")
    print("Run ID:", clean_xgb_run_id_finops)

    # --------------------------------------------------------
    # Reload stored artifact and verify predictions
    # --------------------------------------------------------

    reloaded_xgb_finops = (
        mlflow.xgboost.load_model(
            xgb_logged_model_info_finops.model_uri
        )
    )

    reloaded_xgb_predictions_finops = (
        reloaded_xgb_finops.predict(
            X_test_finops.astype("float64")
        )
    )

    assert np.allclose(
        reloaded_xgb_predictions_finops,
        xgb_test_predictions_finops,
        rtol=1e-6,
        atol=1e-6
    )

    print(
        "Stored XGBoost artifact reloaded successfully."
    )

    print(
        "Reloaded predictions match the "
        "original predictions."
    )

print(
    "XGBoost test MAE:",
    round(xgb_test_metrics_finops["MAE"], 4)
)

Clean XGBoost logged.
Run ID: 74b284e76e0a4042b49f8b62bc85ec12
Stored XGBoost artifact reloaded successfully.
Reloaded predictions match the original predictions.
XGBoost test MAE: 10.0239


In [ ]:
# ============================================================
# CELL 18 — LOG CLEAN LIGHTGBM TO MLFLOW
# ============================================================

RUN_NAME_LGBM_FINOPS = "lightgbm_clean"

existing_lgbm_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        f"tags.mlflow.runName = "
        f"'{RUN_NAME_LGBM_FINOPS}'"
    )
)

finished_lgbm_runs_finops = (
    existing_lgbm_runs_finops[
        existing_lgbm_runs_finops["status"]
        == "FINISHED"
    ]
)

if not finished_lgbm_runs_finops.empty:

    clean_lgbm_run_id_finops = (
        finished_lgbm_runs_finops
        .iloc[0]["run_id"]
    )

    print("Clean LightGBM run already exists.")
    print("Run ID:", clean_lgbm_run_id_finops)

else:

    lgbm_input_example_finops = (
        X_train_finops
        .head(5)
        .astype("float64")
        .copy()
    )

    lgbm_signature_finops = infer_signature(
        lgbm_input_example_finops,
        lgbm_finops.predict(
            lgbm_input_example_finops
        )
    )

    lgbm_feature_importance_finops = {
        feature: float(importance)
        for feature, importance in zip(
            FINOPS_FEATURE_COLUMNS,
            lgbm_finops.feature_importances_
        )
    }

    with mlflow.start_run(
        run_name=RUN_NAME_LGBM_FINOPS
    ) as run:

        mlflow.log_params({
            "model_type": "LGBMRegressor",
            "n_estimators": 300,
            "max_depth": 4,
            "num_leaves": 15,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "subsample_freq": 1,
            "colsample_bytree": 0.8,
            "objective": "regression",
            "forecast_horizon": "1_hour",
            "training_samples": 501,
            "validation_samples": 107,
            "test_samples": 109,
            "num_features": 22,
            "dataset_version": (
                "bitbrains_hourly_clean_719_rows"
            )
        })

        mlflow.log_metrics({
            "validation_mae": (
                lgbm_val_metrics_finops["MAE"]
            ),
            "validation_rmse": (
                lgbm_val_metrics_finops["RMSE"]
            ),
            "validation_r2": (
                lgbm_val_metrics_finops["R2"]
            ),
            "test_mae": (
                lgbm_test_metrics_finops["MAE"]
            ),
            "test_rmse": (
                lgbm_test_metrics_finops["RMSE"]
            ),
            "test_r2": (
                lgbm_test_metrics_finops["R2"]
            ),
            "production_baseline_mae": (
                naive_test_mae_finops
            ),
            "mae_improvement_vs_baseline_pct": (
                lgbm_mae_improvement_finops
            )
        })

        mlflow.set_tags({
            "project": "FinOps Cloud Cost Forecasting",
            "model_family": "gradient_boosting",
            "dataset_status": "clean",
            "deployment_status": "rejected",
            "promotion_decision": "rejected",
            "primary_promotion_metric": "test_mae",
            "rejection_reason": (
                "failed_to_beat_naive_baseline_on_test"
            )
        })

        lgbm_logged_model_info_finops = (
            mlflow.lightgbm.log_model(
                lgb_model=lgbm_finops,
                name="model",
                signature=lgbm_signature_finops,
                input_example=lgbm_input_example_finops
            )
        )

        mlflow.log_dict(
            lgbm_feature_importance_finops,
            "metadata/feature_importance.json"
        )

        clean_lgbm_run_id_finops = (
            run.info.run_id
        )

    print("Clean LightGBM logged.")
    print("Run ID:", clean_lgbm_run_id_finops)

    # --------------------------------------------------------
    # Reload and verify stored artifact
    # --------------------------------------------------------

    reloaded_lgbm_finops = (
        mlflow.lightgbm.load_model(
            lgbm_logged_model_info_finops.model_uri
        )
    )

    reloaded_lgbm_predictions_finops = (
        reloaded_lgbm_finops.predict(
            X_test_finops.astype("float64")
        )
    )

    assert np.allclose(
        reloaded_lgbm_predictions_finops,
        lgbm_test_predictions_finops,
        rtol=1e-6,
        atol=1e-6
    )

    print(
        "Stored LightGBM artifact reloaded successfully."
    )

    print(
        "Reloaded predictions match the "
        "original predictions."
    )

print(
    "LightGBM test MAE:",
    round(lgbm_test_metrics_finops["MAE"], 4)
)

Clean LightGBM logged.
Run ID: 4205ef82beea496f92b4e7916b0d9735
Stored LightGBM artifact reloaded successfully.
Reloaded predictions match the original predictions.
LightGBM test MAE: 9.205


In [ ]:
# ============================================================
# CELL 19 — LOG CLEAN CATBOOST TO MLFLOW
# ============================================================

RUN_NAME_CATBOOST_FINOPS = "catboost_clean"

existing_catboost_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        f"tags.mlflow.runName = "
        f"'{RUN_NAME_CATBOOST_FINOPS}'"
    )
)

finished_catboost_runs_finops = (
    existing_catboost_runs_finops[
        existing_catboost_runs_finops["status"]
        == "FINISHED"
    ]
)

if not finished_catboost_runs_finops.empty:

    clean_catboost_run_id_finops = (
        finished_catboost_runs_finops
        .iloc[0]["run_id"]
    )

    print("Clean CatBoost run already exists.")
    print("Run ID:", clean_catboost_run_id_finops)

else:

    catboost_input_example_finops = (
        X_train_finops
        .head(5)
        .astype("float64")
        .copy()
    )

    catboost_signature_finops = infer_signature(
        catboost_input_example_finops,
        catboost_finops.predict(
            catboost_input_example_finops
        )
    )

    catboost_feature_importance_finops = {
        feature: float(importance)
        for feature, importance in zip(
            FINOPS_FEATURE_COLUMNS,
            catboost_finops.get_feature_importance()
        )
    }

    with mlflow.start_run(
        run_name=RUN_NAME_CATBOOST_FINOPS
    ) as run:

        mlflow.log_params({
            "model_type": "CatBoostRegressor",
            "iterations": 300,
            "depth": 4,
            "learning_rate": 0.05,
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "forecast_horizon": "1_hour",
            "training_samples": 501,
            "validation_samples": 107,
            "test_samples": 109,
            "num_features": 22,
            "dataset_version": (
                "bitbrains_hourly_clean_719_rows"
            )
        })

        mlflow.log_metrics({
            "validation_mae": (
                catboost_val_metrics_finops["MAE"]
            ),
            "validation_rmse": (
                catboost_val_metrics_finops["RMSE"]
            ),
            "validation_r2": (
                catboost_val_metrics_finops["R2"]
            ),
            "test_mae": (
                catboost_test_metrics_finops["MAE"]
            ),
            "test_rmse": (
                catboost_test_metrics_finops["RMSE"]
            ),
            "test_r2": (
                catboost_test_metrics_finops["R2"]
            ),
            "production_baseline_mae": (
                naive_test_mae_finops
            ),
            "mae_improvement_vs_baseline_pct": (
                catboost_mae_improvement_finops
            )
        })

        mlflow.set_tags({
            "project": "FinOps Cloud Cost Forecasting",
            "model_family": "gradient_boosting",
            "dataset_status": "clean",
            "candidate_rank": "best_learned_model",
            "deployment_status": "rejected",
            "promotion_decision": "rejected",
            "primary_promotion_metric": "test_mae",
            "rejection_reason": (
                "failed_to_beat_naive_baseline_on_test"
            )
        })

        catboost_logged_model_info_finops = (
            mlflow.catboost.log_model(
                cb_model=catboost_finops,
                name="model",
                signature=catboost_signature_finops,
                input_example=catboost_input_example_finops
            )
        )

        mlflow.log_dict(
            catboost_feature_importance_finops,
            "metadata/feature_importance.json"
        )

        clean_catboost_run_id_finops = (
            run.info.run_id
        )

    print("Clean CatBoost logged.")
    print("Run ID:", clean_catboost_run_id_finops)

    # --------------------------------------------------------
    # Reload and verify stored artifact
    # --------------------------------------------------------

    reloaded_catboost_finops = (
        mlflow.catboost.load_model(
            catboost_logged_model_info_finops.model_uri
        )
    )

    reloaded_catboost_predictions_finops = (
        reloaded_catboost_finops.predict(
            X_test_finops.astype("float64")
        )
    )

    assert np.allclose(
        reloaded_catboost_predictions_finops,
        catboost_test_predictions_finops,
        rtol=1e-6,
        atol=1e-6
    )

    print(
        "Stored CatBoost artifact reloaded successfully."
    )

    print(
        "Reloaded predictions match the "
        "original predictions."
    )

print(
    "CatBoost test MAE:",
    round(catboost_test_metrics_finops["MAE"], 4)
)

Clean CatBoost logged.
Run ID: 613e2204117e4988b69c676e64bbe501
Stored CatBoost artifact reloaded successfully.
Reloaded predictions match the original predictions.
CatBoost test MAE: 8.5622


In [ ]:
# ============================================================
# CELL 20 — LOG DRIFT-RETRAINED XGBOOST
# ============================================================

RUN_NAME_RETRAINED_FINOPS = (
    "drift_retrained_xgboost_clean"
)

existing_retrained_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ],
    filter_string=(
        f"tags.mlflow.runName = "
        f"'{RUN_NAME_RETRAINED_FINOPS}'"
    )
)

finished_retrained_runs_finops = (
    existing_retrained_runs_finops[
        existing_retrained_runs_finops["status"]
        == "FINISHED"
    ]
)

if not finished_retrained_runs_finops.empty:

    clean_retrained_xgb_run_id_finops = (
        finished_retrained_runs_finops
        .iloc[0]["run_id"]
    )

    print("Drift-retrained XGBoost already exists.")
    print(
        "Run ID:",
        clean_retrained_xgb_run_id_finops
    )

else:

    retrained_input_example_finops = (
        X_retrain_finops
        .head(5)
        .astype("float64")
        .copy()
    )

    retrained_signature_finops = infer_signature(
        retrained_input_example_finops,
        retrained_xgb_finops.predict(
            retrained_input_example_finops
        )
    )

    retrained_feature_importance_finops = {
        feature: float(importance)
        for feature, importance in zip(
            FINOPS_FEATURE_COLUMNS,
            retrained_xgb_finops.feature_importances_
        )
    }

    with mlflow.start_run(
        run_name=RUN_NAME_RETRAINED_FINOPS
    ) as run:

        mlflow.log_params({
            "model_type": "XGBRegressor",
            "training_type": (
                "drift_triggered_retraining"
            ),
            "n_estimators": 300,
            "max_depth": 4,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "forecast_horizon": "1_hour",
            "training_samples": 608,
            "test_samples": 109,
            "num_features": 22,
            "dataset_version": (
                "bitbrains_hourly_clean_719_rows"
            )
        })

        mlflow.log_metrics({
            "test_mae": (
                retrained_xgb_test_metrics_finops["MAE"]
            ),
            "test_rmse": (
                retrained_xgb_test_metrics_finops["RMSE"]
            ),
            "test_r2": (
                retrained_xgb_test_metrics_finops["R2"]
            ),
            "original_xgboost_test_mae": (
                xgb_test_metrics_finops["MAE"]
            ),
            "production_baseline_mae": (
                naive_test_mae_finops
            ),
            "improvement_vs_original_xgboost_pct": (
                retrained_improvement_vs_original_finops
            ),
            "improvement_vs_baseline_pct": (
                retrained_improvement_vs_baseline_finops
            )
        })

        mlflow.set_tags({
            "project": "FinOps Cloud Cost Forecasting",
            "model_family": "gradient_boosting",
            "dataset_status": "clean",
            "retraining_trigger": (
                "temporal_distribution_shift_and_"
                "performance_degradation"
            ),
            "deployment_status": "rejected",
            "promotion_decision": "rejected",
            "primary_promotion_metric": "test_mae",
            "rejection_reason": (
                "retrained_candidate_failed_to_"
                "beat_production_baseline"
            )
        })

        retrained_logged_model_info_finops = (
            mlflow.xgboost.log_model(
                xgb_model=retrained_xgb_finops,
                name="model",
                signature=retrained_signature_finops,
                input_example=(
                    retrained_input_example_finops
                )
            )
        )

        mlflow.log_dict(
            retrained_feature_importance_finops,
            "metadata/feature_importance.json"
        )

        clean_retrained_xgb_run_id_finops = (
            run.info.run_id
        )

    print("Drift-retrained XGBoost logged.")
    print(
        "Run ID:",
        clean_retrained_xgb_run_id_finops
    )

    reloaded_retrained_xgb_finops = (
        mlflow.xgboost.load_model(
            retrained_logged_model_info_finops.model_uri
        )
    )

    reloaded_retrained_predictions_finops = (
        reloaded_retrained_xgb_finops.predict(
            X_test_finops.astype("float64")
        )
    )

    assert np.allclose(
        reloaded_retrained_predictions_finops,
        retrained_xgb_test_predictions_finops,
        rtol=1e-6,
        atol=1e-6
    )

    print(
        "Stored retrained artifact reloaded successfully."
    )
    print(
        "Reloaded predictions match original predictions."
    )

print(
    "Retrained XGBoost test MAE:",
    round(
        retrained_xgb_test_metrics_finops["MAE"],
        4
    )
)

Drift-retrained XGBoost logged.
Run ID: 0790c01add1d4c04912cea3e037d183a
Stored retrained artifact reloaded successfully.
Reloaded predictions match original predictions.
Retrained XGBoost test MAE: 9.1266


In [ ]:
# ============================================================
# CELL 21 — VERIFY COMPLETE CLEAN EXPERIMENT
# ============================================================

all_clean_runs_finops = mlflow.search_runs(
    experiment_ids=[
        clean_experiment_finops.experiment_id
    ]
)

finished_clean_runs_finops = (
    all_clean_runs_finops[
        all_clean_runs_finops["status"]
        == "FINISHED"
    ]
    .copy()
)

comparison_columns_finops = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.validation_mae",
    "metrics.test_mae",
    "metrics.test_rmse",
    "metrics.test_r2",
    "metrics.mae_improvement_vs_baseline_pct",
    "tags.deployment_status",
    "tags.promotion_decision"
]

comparison_columns_finops = [
    column
    for column in comparison_columns_finops
    if column in finished_clean_runs_finops.columns
]

clean_mlflow_comparison_finops = (
    finished_clean_runs_finops[
        comparison_columns_finops
    ]
    .sort_values(
        "metrics.test_mae",
        ascending=True
    )
    .reset_index(drop=True)
)

print("FINAL CLEAN MLFLOW COMPARISON")
print("=" * 100)

display(
    clean_mlflow_comparison_finops
)

expected_run_names_finops = {
    "naive_baseline_clean",
    "xgboost_clean",
    "lightgbm_clean",
    "catboost_clean",
    "drift_retrained_xgboost_clean"
}

actual_run_names_finops = set(
    finished_clean_runs_finops[
        "tags.mlflow.runName"
    ]
)

missing_runs_finops = (
    expected_run_names_finops
    - actual_run_names_finops
)

print("Finished clean runs:", len(finished_clean_runs_finops))
print("Missing expected runs:", missing_runs_finops)

assert not missing_runs_finops
assert len(finished_clean_runs_finops) == 5

# Refresh run IDs directly from verified MLflow data
run_id_by_name_finops = dict(zip(
    finished_clean_runs_finops[
        "tags.mlflow.runName"
    ],
    finished_clean_runs_finops["run_id"]
))

clean_naive_run_id_finops = (
    run_id_by_name_finops["naive_baseline_clean"]
)

clean_xgb_run_id_finops = (
    run_id_by_name_finops["xgboost_clean"]
)

clean_lgbm_run_id_finops = (
    run_id_by_name_finops["lightgbm_clean"]
)

clean_catboost_run_id_finops = (
    run_id_by_name_finops["catboost_clean"]
)

clean_retrained_xgb_run_id_finops = (
    run_id_by_name_finops[
        "drift_retrained_xgboost_clean"
    ]
)

best_mlflow_run_name_finops = (
    clean_mlflow_comparison_finops
    .iloc[0]["tags.mlflow.runName"]
)

assert (
    best_mlflow_run_name_finops
    == "naive_baseline_clean"
)

print("\nClean MLflow experiment verification passed.")
print(
    "Production winner:",
    best_mlflow_run_name_finops
)

FINAL CLEAN MLFLOW COMPARISON


,run_id,tags.mlflow.runName,metrics.validation_mae,metrics.test_mae,metrics.test_rmse,metrics.test_r2,metrics.mae_improvement_vs_baseline_pct,tags.deployment_status,tags.promotion_decision
0,25b920c78baf4274a36b9cea73f39e23,naive_baseline_clean,5.845166,7.882846,10.864452,0.117757,NaN,production,retained
1,613e2204117e4988b69c676e64bbe501,catboost_clean,5.698595,8.562205,10.669091,0.149200,-8.618188,rejected,rejected
2,0790c01add1d4c04912cea3e037d183a,drift_retrained_xgboost_clean,NaN,9.126611,10.730183,0.139428,NaN,rejected,rejected
3,4205ef82beea496f92b4e7916b0d9735,lightgbm_clean,6.222403,9.205031,11.595692,-0.005000,-16.772937,rejected,rejected
4,74b284e76e0a4042b49f8b62bc85ec12,xgboost_clean,5.742888,10.023914,12.057969,-0.086728,-27.161101,rejected,rejected


Finished clean runs: 5
Missing expected runs: set()

Clean MLflow experiment verification passed.
Production winner: naive_baseline_clean


In [ ]:
# ============================================================
# CELL 22 — MLFLOW MODEL REGISTRY
# ============================================================

from mlflow.tracking import MlflowClient

mlflow.set_registry_uri(
    MLFLOW_TRACKING_URI_FINOPS
)

registry_client_finops = MlflowClient(
    tracking_uri=MLFLOW_TRACKING_URI_FINOPS,
    registry_uri=MLFLOW_TRACKING_URI_FINOPS
)

REGISTERED_MODEL_NAME_FINOPS = (
    "finops-cloud-cost-forecasting-clean-v1"
)

# ------------------------------------------------------------
# Idempotent registration helper
# ------------------------------------------------------------

def get_or_register_version_finops(
    registered_model_name,
    source_run_id
):
    try:
        existing_versions = (
            registry_client_finops.search_model_versions(
                f"name='{registered_model_name}'"
            )
        )
    except Exception:
        existing_versions = []

    for version in existing_versions:
        if version.run_id == source_run_id:
            print(
                "Existing registered version found:",
                version.version
            )
            return version

    model_uri = (
        f"runs:/{source_run_id}/model"
    )

    registered_version = mlflow.register_model(
        model_uri=model_uri,
        name=registered_model_name
    )

    print(
        "New registered version created:",
        registered_version.version
    )

    return registered_version


# ------------------------------------------------------------
# Register production baseline
# ------------------------------------------------------------

champion_version_finops = (
    get_or_register_version_finops(
        REGISTERED_MODEL_NAME_FINOPS,
        clean_naive_run_id_finops
    )
)

# ------------------------------------------------------------
# Register best learned candidate
# ------------------------------------------------------------

challenger_version_finops = (
    get_or_register_version_finops(
        REGISTERED_MODEL_NAME_FINOPS,
        clean_catboost_run_id_finops
    )
)

# ------------------------------------------------------------
# Apply aliases
# ------------------------------------------------------------

registry_client_finops.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME_FINOPS,
    alias="champion",
    version=champion_version_finops.version
)

registry_client_finops.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME_FINOPS,
    alias="challenger",
    version=challenger_version_finops.version
)

# ------------------------------------------------------------
# Add version metadata
# ------------------------------------------------------------

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=champion_version_finops.version,
    key="lifecycle_role",
    value="production_champion"
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=champion_version_finops.version,
    key="test_mae",
    value=f"{naive_test_mae_finops:.6f}"
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=challenger_version_finops.version,
    key="lifecycle_role",
    value="best_learned_challenger"
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=challenger_version_finops.version,
    key="promotion_decision",
    value="rejected"
)

registry_client_finops.set_model_version_tag(
    name=REGISTERED_MODEL_NAME_FINOPS,
    version=challenger_version_finops.version,
    key="test_mae",
    value=f"{catboost_test_metrics_finops['MAE']:.6f}"
)

registry_client_finops.update_registered_model(
    name=REGISTERED_MODEL_NAME_FINOPS,
    description=(
        "One-hour FinOps estimated-cost forecasting model. "
        "The champion alias points to the production "
        "persistence baseline. The challenger alias points "
        "to CatBoost, the best learned model, which remains "
        "rejected until it beats the champion on test MAE."
    )
)

# ------------------------------------------------------------
# Verify registry
# ------------------------------------------------------------

champion_alias_finops = (
    registry_client_finops.get_model_version_by_alias(
        REGISTERED_MODEL_NAME_FINOPS,
        "champion"
    )
)

challenger_alias_finops = (
    registry_client_finops.get_model_version_by_alias(
        REGISTERED_MODEL_NAME_FINOPS,
        "challenger"
    )
)

print("MODEL REGISTRY CREATED")
print("=" * 70)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME_FINOPS
)

print(
    "Champion version:",
    champion_alias_finops.version
)

print(
    "Champion source run:",
    champion_alias_finops.run_id
)

print(
    "Challenger version:",
    challenger_alias_finops.version
)

print(
    "Challenger source run:",
    challenger_alias_finops.run_id
)

assert (
    champion_alias_finops.run_id
    == clean_naive_run_id_finops
)

assert (
    challenger_alias_finops.run_id
    == clean_catboost_run_id_finops
)

print("\nModel Registry verification passed.")

Successfully registered model 'finops-cloud-cost-forecasting-clean-v1'.
2026/08/24 05:16:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 25b920c78baf4274a36b9cea73f39e23 has no artifacts at artifact path 'model', registering model based on models:/m-36ed6d62987f4c1892a1a554a97c1879 instead
Created version '1' of model 'finops-cloud-cost-forecasting-clean-v1'.
Registered model 'finops-cloud-cost-forecasting-clean-v1' already exists. Creating a new version of this model...
2026/08/24 05:16:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 613e2204117e4988b69c676e64bbe501 has no artifacts at artifact path 'model', registering model based on models:/m-e7e506ed984645e5b792722a4332f535 instead


New registered version created: 1


Created version '2' of model 'finops-cloud-cost-forecasting-clean-v1'.


New registered version created: 2
MODEL REGISTRY CREATED
Registered model: finops-cloud-cost-forecasting-clean-v1
Champion version: 1
Champion source run: 25b920c78baf4274a36b9cea73f39e23
Challenger version: 2
Challenger source run: 613e2204117e4988b69c676e64bbe501

Model Registry verification passed.


In [ ]:
# ============================================================
# CELL 23 — REGISTRY ALIAS INFERENCE SMOKE TEST
# ============================================================

CHAMPION_MODEL_URI_FINOPS = (
    f"models:/{REGISTERED_MODEL_NAME_FINOPS}@champion"
)

CHALLENGER_MODEL_URI_FINOPS = (
    f"models:/{REGISTERED_MODEL_NAME_FINOPS}@challenger"
)

print("Loading registry aliases...")

champion_model_finops = mlflow.pyfunc.load_model(
    CHAMPION_MODEL_URI_FINOPS
)

challenger_model_finops = mlflow.pyfunc.load_model(
    CHALLENGER_MODEL_URI_FINOPS
)

# Champion baseline requires only current cost
champion_input_finops = (
    X_test_finops[
        ["estimated_cost_index"]
    ]
    .astype("float64")
    .copy()
)

# CatBoost challenger requires all 22 features
challenger_input_finops = (
    X_test_finops[
        FINOPS_FEATURE_COLUMNS
    ]
    .astype("float64")
    .copy()
)

champion_registry_predictions_finops = np.asarray(
    champion_model_finops.predict(
        champion_input_finops
    )
).reshape(-1)

challenger_registry_predictions_finops = np.asarray(
    challenger_model_finops.predict(
        challenger_input_finops
    )
).reshape(-1)

# ------------------------------------------------------------
# Verify predictions against original evaluated models
# ------------------------------------------------------------

assert np.allclose(
    champion_registry_predictions_finops,
    naive_test_predictions_finops,
    rtol=1e-6,
    atol=1e-6
)

assert np.allclose(
    challenger_registry_predictions_finops,
    catboost_test_predictions_finops,
    rtol=1e-6,
    atol=1e-6
)

champion_registry_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        champion_registry_predictions_finops
    )
)

challenger_registry_metrics_finops = (
    calculate_regression_metrics_finops(
        y_test_finops,
        challenger_registry_predictions_finops
    )
)

registry_smoke_test_finops = pd.DataFrame([
    {
        "Alias": "champion",
        "Model": "Naive Baseline",
        **champion_registry_metrics_finops
    },
    {
        "Alias": "challenger",
        "Model": "CatBoost",
        **challenger_registry_metrics_finops
    }
])

print("\nMODEL REGISTRY INFERENCE RESULTS")
print("=" * 75)

display(
    registry_smoke_test_finops.round(4)
)

print("First 5 registry predictions:")

display(
    pd.DataFrame({
        "actual_next_hour_cost": (
            y_test_finops.iloc[:5].to_numpy()
        ),
        "champion_prediction": (
            champion_registry_predictions_finops[:5]
        ),
        "challenger_prediction": (
            challenger_registry_predictions_finops[:5]
        )
    })
)

assert np.isclose(
    champion_registry_metrics_finops["MAE"],
    naive_test_mae_finops,
    atol=1e-6
)

assert np.isclose(
    challenger_registry_metrics_finops["MAE"],
    catboost_test_metrics_finops["MAE"],
    atol=1e-6
)

print("\nRegistry aliases loaded successfully.")
print("Champion predictions verified.")
print("Challenger predictions verified.")

print(
    "\nNOTEBOOK 04 COMPLETE — "
    "MLflow tracking and Model Registry are ready."
)

Loading registry aliases...

MODEL REGISTRY INFERENCE RESULTS


,Alias,Model,MAE,RMSE,R2
0,champion,Naive Baseline,7.8828,10.8645,0.1178
1,challenger,CatBoost,8.5622,10.6691,0.1492


First 5 registry predictions:


,actual_next_hour_cost,champion_prediction,challenger_prediction
0,32.037608,21.629732,31.402184
1,40.416795,32.037608,32.011261
2,19.763919,40.416795,38.417573
3,30.830895,19.763919,24.024167
4,22.854520,30.830895,32.976832



Registry aliases loaded successfully.
Champion predictions verified.
Challenger predictions verified.

NOTEBOOK 04 COMPLETE — MLflow tracking and Model Registry are ready.


In [ ]:
# ============================================================
# EXPORT TRAINING REFERENCE PROFILE FOR DRIFT MONITORING
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Confirm that the training data exists
# ------------------------------------------------------------

if "X_train_finops" not in globals():
    raise RuntimeError(
        "X_train_finops is unavailable. Run the Notebook 04 "
        "cells that load the data and create the train/test "
        "split before running this cell."
    )


FEATURE_NAME_FINOPS = "estimated_cost_index"

if FEATURE_NAME_FINOPS not in X_train_finops.columns:
    raise KeyError(
        f"Training feature not found: {FEATURE_NAME_FINOPS}"
    )


# ------------------------------------------------------------
# 2. Clean the reference feature
# ------------------------------------------------------------

training_values_finops = pd.to_numeric(
    X_train_finops[FEATURE_NAME_FINOPS],
    errors="coerce"
).to_numpy(dtype="float64")

training_values_finops = training_values_finops[
    np.isfinite(training_values_finops)
]

if len(training_values_finops) < 30:
    raise RuntimeError(
        "At least 30 valid training values are required "
        "to build a reliable reference profile."
    )


# ------------------------------------------------------------
# 3. Create quantile-based PSI bins
# ------------------------------------------------------------

NUMBER_OF_BINS_FINOPS = 10

quantile_points_finops = np.linspace(
    0.0,
    1.0,
    NUMBER_OF_BINS_FINOPS + 1
)

quantile_edges_finops = np.quantile(
    training_values_finops,
    quantile_points_finops
)

unique_edges_finops = np.unique(
    quantile_edges_finops
)

# A nearly constant feature needs special boundaries.
if len(unique_edges_finops) < 3:

    reference_center_finops = float(
        np.mean(training_values_finops)
    )

    reference_width_finops = max(
        abs(reference_center_finops) * 0.01,
        1e-6
    )

    internal_bin_edges_finops = np.array([
        reference_center_finops
        - reference_width_finops,

        reference_center_finops
        + reference_width_finops
    ])

else:

    # The minimum and maximum are removed because production
    # values may fall outside the training range.
    internal_bin_edges_finops = (
        unique_edges_finops[1:-1]
    )


histogram_edges_finops = np.concatenate([
    [-np.inf],
    internal_bin_edges_finops,
    [np.inf]
])

reference_counts_finops, _ = np.histogram(
    training_values_finops,
    bins=histogram_edges_finops
)

reference_proportions_finops = (
    reference_counts_finops
    / reference_counts_finops.sum()
)


# ------------------------------------------------------------
# 4. Build the portable reference profile
# ------------------------------------------------------------

reference_profile_finops = {
    "schema_version": "1.0",
    "created_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "registered_model": (
        "finops-cloud-cost-forecasting-clean-v1"
    ),
    "model_alias": "champion",
    "model_version": "1",
    "minimum_production_samples": 30,
    "psi_thresholds": {
        "moderate": 0.10,
        "significant": 0.25
    },
    "features": {
        FEATURE_NAME_FINOPS: {
            "dtype": "float64",
            "training_sample_count": int(
                len(training_values_finops)
            ),
            "minimum": float(
                np.min(training_values_finops)
            ),
            "maximum": float(
                np.max(training_values_finops)
            ),
            "mean": float(
                np.mean(training_values_finops)
            ),
            "standard_deviation": float(
                np.std(training_values_finops)
            ),
            "internal_bin_edges": (
                internal_bin_edges_finops.tolist()
            ),
            "expected_bin_proportions": (
                reference_proportions_finops.tolist()
            )
        }
    }
}


# ------------------------------------------------------------
# 5. Save to Google Drive
# ------------------------------------------------------------

REFERENCE_EXPORT_DIRECTORY_FINOPS = Path(
    "/content/drive/MyDrive/finops_deployment_exports"
)

REFERENCE_EXPORT_DIRECTORY_FINOPS.mkdir(
    parents=True,
    exist_ok=True
)

REFERENCE_PROFILE_PATH_FINOPS = (
    REFERENCE_EXPORT_DIRECTORY_FINOPS
    / "reference_profile.json"
)

with REFERENCE_PROFILE_PATH_FINOPS.open(
    "w",
    encoding="utf-8"
) as reference_file_finops:

    json.dump(
        reference_profile_finops,
        reference_file_finops,
        indent=2
    )


# ------------------------------------------------------------
# 6. Verify the generated file
# ------------------------------------------------------------

assert REFERENCE_PROFILE_PATH_FINOPS.exists()

assert np.isclose(
    np.sum(reference_proportions_finops),
    1.0
)

print("TRAINING REFERENCE PROFILE CREATED")
print("=" * 70)

print(
    "Feature:",
    FEATURE_NAME_FINOPS
)

print(
    "Training samples:",
    len(training_values_finops)
)

print(
    "Number of PSI bins:",
    len(reference_proportions_finops)
)

print(
    "Expected proportions sum:",
    reference_proportions_finops.sum()
)

print(
    "Saved to:",
    REFERENCE_PROFILE_PATH_FINOPS
)

TRAINING REFERENCE PROFILE CREATED
Feature: estimated_cost_index
Training samples: 501
Number of PSI bins: 10
Expected proportions sum: 1.0
Saved to: /content/drive/MyDrive/finops_deployment_exports/reference_profile.json
